Here we will try to make use of the extracted type 2,4,5 data from the rsp_events file.  

In [6]:
import json
import numpy as np

# Load the JSON file
with open("Fetch data from NOAA/rsp_events.json", "r") as f:
    data = json.load(f)

# Lists to hold each burst type
type_II = []
type_IV = []
type_V = []

# Loop through each event and classify by burst type in "part"
for event in data:
    part = event.get("part", "")
    if part.startswith("II/"):
        type_II.append(event)
    elif part.startswith("IV/"):
        type_IV.append(event)
    elif part.startswith("V/"):
        type_V.append(event)

# Convert to numpy arrays for shape
type_II = np.array(type_II)
type_IV = np.array(type_IV)
type_V = np.array(type_V)

print("Type II shape:", type_II.shape)
print("Type IV shape:", type_IV.shape)
print("Type V shape:", type_V.shape)

Type II shape: (179,)
Type IV shape: (250,)
Type V shape: (284,)


Since most of the data in the offline dataset is not labelled. We will now try to find the find and label the data indexes which are in the offline data from the help of the rsp file. We will save the indexes of the offline data file which does not have corresponding file in the rsp. We will append the h5 file in labels keys with the type from rsp. To append the labels key the condition that the data should not have corresponding label in the h5 file should be met. If the h5 file has label for an image and also for the rsp file, the label would not be replaced. The file will checked via date and time from rsp and timestamp key from the h5 file. The indexes of timestamps for which the labels were appended will also be saved in another variable. But the not found timestamps indexes will be saved in a seperate json file. 

In [2]:
import json

# Load the JSON file
with open("rsp_events.json", "r") as f:
    data = json.load(f)

date_begin_to_part = {}

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("part")
    if date and begin and part and begin != "////":
        # Format begin as HH:MM:00.000000
        hour = begin[:2]
        minute = begin[2:]
        formatted = f"{date}_{hour}:{minute}:00.000000"
        # Drop the number after /
        part_clean = part.split('/')[0]
        date_begin_to_part[formatted] = part_clean

# Example: print first 10 entries
for k, v in list(date_begin_to_part.items())[:10]:
    print(f"{k}: {v}")

print("Length of dictionary:", len(date_begin_to_part))

2013-03-19_00:39:00.000000: IV
2013-04-06_05:58:00.000000: V
2013-04-11_09:39:00.000000: V
2013-04-11_10:20:00.000000: IV
2013-04-18_07:59:00.000000: V
2013-04-22_10:26:00.000000: V
2013-04-22_20:44:00.000000: V
2013-04-22_21:16:00.000000: V
2013-04-22_22:20:00.000000: V
2013-04-22_22:40:00.000000: V
Length of dictionary: 697


In [3]:
import json

with open("rsp_events.json", "r") as f:
    data = json.load(f)

key_counts = {}
for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("part")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        key = f"{date}_{hour}:{minute}:00.000000"
        key_counts[key] = key_counts.get(key, 0) + 1

duplicates = {k: v for k, v in key_counts.items() if v > 1}
print(f"Number of duplicate keys: {len(duplicates)}")
for k, v in duplicates.items():
    print(f"{k}: {v} entries")

Number of duplicate keys: 17
2013-04-22_22:20:00.000000: 2 entries
2013-05-31_19:57:00.000000: 2 entries
2013-06-03_00:00:00.000000: 2 entries
2013-10-28_04:37:00.000000: 2 entries
2014-02-17_03:00:00.000000: 2 entries
2014-07-04_04:38:00.000000: 2 entries
2014-08-01_18:18:00.000000: 2 entries
2014-11-03_03:48:00.000000: 2 entries
2015-06-24_00:00:00.000000: 2 entries
2016-01-01_23:24:00.000000: 2 entries
2016-02-03_23:25:00.000000: 2 entries
2022-04-30_09:57:00.000000: 2 entries
2022-08-27_02:12:00.000000: 2 entries
2022-09-23_13:50:00.000000: 2 entries
2024-01-28_02:28:00.000000: 2 entries
2024-02-06_03:10:00.000000: 2 entries
2024-07-21_01:53:00.000000: 2 entries


The reason you have 714 entries in your JSON file but only 697 unique keys in your dictionary is because some entries have the same combination of date and begin (i.e., the same timestamp key). When you use a dictionary, if two or more events have the same date and begin, the last one will overwrite the previous ones.

Some of the duplicates have same same types but some have different types of burst at the same time instant. Eg, 2013-05-31_19:57:00.000000: 2 entries

Merging part (bursts) of duplicate keys with a "/". These images have 2 bursts for 1.

In [27]:
import json

# Load the JSON file
with open("rsp_events.json", "r") as f:
    data = json.load(f)

roman_to_int = {"II": "2", "IV": "4", "V": "5"}

date_begin_to_parts = {}

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("part")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        key = f"{date}_{hour}:{minute}:00.000000"
        # Drop the number after /
        part_clean = part.split('/')[0]
        # Replace Roman numeral with integer
        for roman, integer in roman_to_int.items():
            if part_clean == roman:
                part_clean = integer
        if key in date_begin_to_parts:
            existing = date_begin_to_parts[key].split('/')
            if part_clean not in existing:
                date_begin_to_parts[key] += f"/{part_clean}"
        else:
            date_begin_to_parts[key] = part_clean

print("json dict:",(date_begin_to_parts))

json dict: {'2013-03-19_00:39:00.000000': '4', '2013-04-06_05:58:00.000000': '5', '2013-04-11_09:39:00.000000': '5', '2013-04-11_10:20:00.000000': '4', '2013-04-18_07:59:00.000000': '5', '2013-04-22_10:26:00.000000': '5', '2013-04-22_20:44:00.000000': '5', '2013-04-22_21:16:00.000000': '5', '2013-04-22_22:20:00.000000': '5', '2013-04-22_22:40:00.000000': '5', '2013-04-23_01:58:00.000000': '5', '2013-04-23_05:08:00.000000': '5', '2013-04-23_13:10:00.000000': '5', '2013-04-23_15:05:00.000000': '5', '2013-05-03_17:56:00.000000': '2', '2013-05-03_18:47:00.000000': '4', '2013-05-12_01:01:00.000000': '5', '2013-05-13_15:57:00.000000': '4', '2013-05-14_01:13:00.000000': '4', '2013-05-21_11:41:00.000000': '5', '2013-05-25_06:02:00.000000': '5', '2013-05-25_14:08:00.000000': '5', '2013-05-25_14:09:00.000000': '5', '2013-05-31_19:57:00.000000': '2/4', '2013-06-02_23:15:00.000000': '4', '2013-06-02_23:51:00.000000': '2', '2013-06-03_00:00:00.000000': '4/2', '2013-06-03_03:50:00.000000': '4', '201

In above I was having problem with finding overlapping keys because, the timestamps in offile dataset has timestamps minutes as intervals of 15 (eg, 10:00, 10:15, 10:30), and the rsp has exact moments of the bursts which is not divided into intervals of 15 minutes

Make a variable of the timestamps which exists in both the data.

In [30]:
import h5py
import re
import json
from datetime import datetime, timedelta

# Function to approximate minutes to nearest 15-min interval
def approximate_minutes(timestamp):
    # Extract date and time parts
    parts = timestamp.split('_')
    date_part = parts[0]
    time_parts = parts[1].split(':')
    hour = int(time_parts[0])
    minute = int(time_parts[1])
    
    # Round to nearest 15 minutes
    if minute < 7.5:
        new_minute = 0
        new_hour = hour
    elif minute < 22.5:
        new_minute = 15
        new_hour = hour
    elif minute < 37.5:
        new_minute = 30
        new_hour = hour
    elif minute < 52.5:
        new_minute = 45
        new_hour = hour
    else:
        new_minute = 0
        new_hour = (hour + 1) % 24  # Handle hour rollover
        if new_hour == 0 and hour == 23:
            # This would be a date rollover - we'll ignore this edge case for now
            pass
    
    # Format the new timestamp
    return f"{date_part}_{new_hour:02d}:{new_minute:02d}:00.000000"

# Load the dictionary with burst types
with open("rsp_events.json", "r") as f:
    data = json.load(f)

# Create the dictionary with approximated times
roman_to_int = {"II": "2", "IV": "4", "V": "5"}
date_begin_to_parts = {}
approx_to_original = {}  # To keep track of mapping

for event in data:
    date = event.get("date")
    begin = event.get("begin")
    part = event.get("part")
    if date and begin and part and begin != "////":
        hour = begin[:2]
        minute = begin[2:]
        original_key = f"{date}_{hour}:{minute}:00.000000"
        # Approximate the time
        approx_key = approximate_minutes(original_key)
        # Store mapping from approximated to original
        approx_to_original[approx_key] = original_key
        
        part_clean = part.split('/')[0]
        for roman, integer in roman_to_int.items():
            if part_clean == roman:
                part_clean = integer
                
        if approx_key in date_begin_to_parts:
            existing = date_begin_to_parts[approx_key].split('/')
            if part_clean not in existing:
                date_begin_to_parts[approx_key] += f"/{part_clean}"
        else:
            date_begin_to_parts[approx_key] = part_clean

# Open H5 file and process timestamps with approximation
with h5py.File('/Volumes/External SSD 512gb/dset_with_labels.h5', 'r') as file:
    timestamps = file['timestamps'][:]
    
    # Extract date-time and create approximation mapping
    dt_pattern = re.compile(r'\d{4}-\d{2}-\d{2}_\d{2}:\d{2}:\d{2}\.\d+')
    dt_to_index = {}
    h5_approx_to_original = {}
    
    for i, ts in enumerate(timestamps):
        decoded = ts.decode()
        match = dt_pattern.search(decoded)
        if match:
            original_dt = match.group(0)
            approx_dt = approximate_minutes(original_dt)
            dt_to_index[approx_dt] = i
            h5_approx_to_original[approx_dt] = original_dt
    
    # Find overlapping timestamps using approximated keys
    overlapping_keys = []
    overlapping_indices = []
    overlapping_original_keys = []
    
    for approx_key in date_begin_to_parts:
        if approx_key in dt_to_index:
            overlapping_keys.append(approx_key)
            overlapping_indices.append(dt_to_index[approx_key])
            overlapping_original_keys.append((approx_to_original[approx_key], h5_approx_to_original[approx_key]))
    
    # Find non-overlapping keys
    non_overlapping_keys = [key for key in date_begin_to_parts if key not in dt_to_index]

# Print results
print(f"\nFound {len(overlapping_keys)} timestamps that exist in both sources after approximation")
print(f"First 5 overlapping approximated timestamps: {overlapping_keys[:5]}")
print(f"First 5 corresponding H5 indices: {overlapping_indices[:5]}")

if overlapping_original_keys:
    print("\nSample original timestamp pairs (RSP, H5):")
    for i, (rsp_orig, h5_orig) in enumerate(overlapping_original_keys[:5]):
        print(f"  {i+1}. RSP: {rsp_orig} -> H5: {h5_orig}")

print(f"\nFound {len(non_overlapping_keys)} timestamps in RSP that don't exist in H5")


Found 14 timestamps that exist in both sources after approximation
First 5 overlapping approximated timestamps: ['2022-05-04_10:00:00.000000', '2022-05-04_15:15:00.000000', '2022-05-25_18:30:00.000000', '2022-07-02_16:45:00.000000', '2022-08-03_17:00:00.000000']
First 5 corresponding H5 indices: [108, 101, 1706, 5519, 2279]

Sample original timestamp pairs (RSP, H5):
  1. RSP: 2022-05-04_09:55:00.000000 -> H5: 2022-05-04_10:00:00.000000
  2. RSP: 2022-05-04_15:12:00.000000 -> H5: 2022-05-04_15:15:00.000000
  3. RSP: 2022-05-25_18:24:00.000000 -> H5: 2022-05-25_18:30:00.000000
  4. RSP: 2022-07-02_16:38:00.000000 -> H5: 2022-07-02_16:45:00.000000
  5. RSP: 2022-08-03_17:05:00.000000 -> H5: 2022-08-03_17:00:00.000000

Found 653 timestamps in RSP that don't exist in H5


We only got 14 overlapping timestamps which suggests that the LOFAR data is indeed has european observation and solarmonitor has american observation. So since we have 653 type 2,4,5 we can create artifical dynamic spectrum from this data. We also have the exact time of bursts. But we will not use this data directly as out training set as the data might be missing other noises that we might find in the real life generated dynamic spectrum. So we would use these for GAN and generate synthetic data which is similar to what we have in real life. 

Alternatevily, I would also run this code on unlabelled dataset to find any overlaps. But for this we don not have timestamps to confirm the presence of a burst in the unlabelled dataset. So we might have to use another way to find them. Maybe we can use the patterns of burst we create now from synthetic dataset and try to match it in the unlabeeled dataseet. I am not sure if this would work :(

Import csv file for type 2 for labelling